In this notbook we are going throught the nert architecture and imeplemnt this from scratch. A lot of this code use a transformer. REminder, Bert only uses the encoder from transformer. we are not implementing the transformer blog, you can look at the transformer implementation

In [1]:
import math
import re
from random import * # this will be used to randomly picking the mask tokens
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Data

so for data, this tutorial use sth from local, its a small text from wikipedia just to test, you cna use any data you want to train.

In [ ]:
import spacy  # very simple tockeniser
# TODO:: change this data woth real data
#  TODO:: this example ius use spacy but i think the paper itself use using word2 piece??? not sure need to check
with open("data.txt", "r") as f:
    raw_text = f.read()

# PreProcessing

tokeniszation and numericalization
This step uses spaCy as a simple tokenizer to split the raw text into tokens.

Note: The paper itself uses WordPiece tokenization — spaCy is just a convenient stand-in for learning purposes.

If you get [E050] Can't find model 'en_core_web_sm'
The model needs to be downloaded into the project's virtual environment, not the system Python. Run:
make sure you're using the project venv
.venv/bin/python -m spacy download en_core_web_sm


In [3]:
import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp(raw_text)
sentences = list(doc.sents)


#  lower case and clean all the symbols
text = [x.text.lower() for x in sentences]
text = [re.sub("[.,!?\\-]", '', x) for x in text]

In [4]:
text

['artificial intelligence is a branch of computer science that studies how machines can perform tasks that usually require human intelligence',
 'these tasks include recognizing patterns understanding language making predictions and solving problems\n\n',
 'machine learning is a major area within artificial intelligence',
 'in machine learning a system improves its performance by learning from data rather than following only fixed rules',
 'a model is trained on examples and then used to make decisions on new inputs\n\n',
 'natural language processing focuses on how computers work with human language',
 'it includes tasks such as text classification translation summarization question answering and named entity recognition',
 'modern language models often rely on transformer architectures\n\n',
 'the transformer became important because it replaced recurrence with attention mechanisms',
 'attention allows a model to compare words in a sentence directly even when they are far apart',
 't

In [5]:
# making vocabs - numericalization
word_list= list(set(" ".join(text).split()))
word2id = { '[PAD]': 0, '[CLS]': 1, '[SEP]': 2 , '[MASK]': 3 }

In [6]:
for i, w in enumerate(word_list):
    word2id[w] = i + 4 #reverse the first 0-3 for CLS, PAD - we are going to index from 4
    id2word = {i:w for i, w in enumerate(word2id)}
    vocab_size = len (word2id)

token_list = list ()
for sentence in sentences:
    arr = [word2id[word] for sentence in text for word in sentence.split()]
    token_list.append(arr)

In [7]:
for tokens in token_list[0]:
    print(id2word[tokens])
    print(word2id[id2word[tokens]])

artificial
68
intelligence
50
is
77
a
111
branch
22
of
193
computer
119
science
39
that
76
studies
35
how
127
machines
105
can
149
perform
92
tasks
157
that
76
usually
120
require
164
human
71
intelligence
50
these
159
tasks
157
include
169
recognizing
174
patterns
166
understanding
44
language
142
making
137
predictions
97
and
177
solving
74
problems
80
machine
4
learning
96
is
77
a
111
major
163
area
103
within
81
artificial
68
intelligence
50
in
75
machine
4
learning
96
a
111
system
125
improves
155
its
16
performance
126
by
189
learning
96
from
194
data
167
rather
180
than
150
following
26
only
151
fixed
29
rules
102
a
111
model
28
is
77
trained
178
on
104
examples
66
and
177
then
173
used
7
to
17
make
101
decisions
176
on
104
new
83
inputs
124
natural
99
language
142
processing
93
focuses
185
on
104
how
127
computers
191
work
106
with
64
human
71
language
142
it
172
includes
145
tasks
157
such
11
as
121
text
183
classification
154
translation
160
summarization
5
question
57
answer

# Data Loader

So in the data loader, we want to create a series of pre-processing. So in the bert paper they ahve 2 types of e,nedding, one is token embedding and ither one is segment embedding.
What thye do on Token embeddings - for all sentences we add the [CLS] - which is the classification token, and then [SEP] - whoch is the separator token, which separates between two sentences. for fo the example blow:
'My cat is fluffy. He follows butterflies' would be -> '[CLS] My cat is fluffy. [SEP]. He follows butterflies' - > this is used for next sentence prediction  - NSP
then we have the segment embedding whoch separayes two sentences i.e: [00001111]
then we have masking, as mentioned in bert paper, to be bale to do bidirectioanl learning it needs to randomly assigns 15% of the sequenxes. In this 15%, 80% is replaces with mask, 10% replaces with random tokens (words) and the rest 10% is remain unchanges. 
then last setpis padding once we mask we will add the padding.

varble callwed positive and negative to keep track positive is two sentence is next to another, so we sont want to positive or to negatove, maybe we do half and half

In [8]:
batch_size = 6
max_mask = 5 # calculated roghly of 15%  -> TODO:: mske this dynamix to avg sentence length
max_len = 1000 # maximum length that my transfoemer will expect - this means that our sentence would be too long .. all sentence will be padded

In [9]:
def make_batch():
    batch  = []
    positive = negative = 0
    while positive != batch_size/2 or negative != batch_size/2 :
        # randomly choose two sentence
        tokens_a_index, tokens_b_index = randrange(len(sentences)), randrange(len(sentences))
        tokens_a, tokens_b = token_list[tokens_a_index], token_list[tokens_b_index]

        # 1. token embedding  - add CLS and SEP accordingly
        input_ids = [word2id['[CLS]']] + tokens_a + [word2id['[SEP]']] + tokens_b + [word2id['[SEP]']]

        # 2. create a segment embedding  - which sentence is 0 and 1
        segment_ids = [0] * (1 + len(tokens_a) + 1) + [1] * (len(tokens_b) + 1)

        # 3. masking
        n_pred = min(max_mask, max(1, int(round(len(input_ids) * 0.15))))

        # get all the positions excluding CLS and SEP
        # before i can mask them, I need to know the position - which one I have to mask
        candidates_masked_pos =  [i for i, token in enumerate(input_ids) if token != word2id['[CLS]'] and token != word2id['[SEP]']]
        shuffle(candidates_masked_pos)
        masked_tokens, masked_pos = [], [] # this is going to track which token i masked and with its poisition

        # simply loop and mask accordingly
        for pos in candidates_masked_pos[:n_pred]:
            masked_pos.append(pos)
            masked_tokens.append(input_ids[pos])
            if random () < 0.1:  # 10% replaces with random token
                index = randint(0, vocab_size - 1)
                input_ids[pos] = word2id[id2word[index]]
            elif random() < 0.8: # 80% replace woith mask
                input_ids[pos] = word2id['[MASK]']
            else:
                pass
                        
        # 4.pad the sentence  padding to the max length - input of transformer
        n_pad = max_len - len(input_ids)
        input_ids.extend([0] * n_pad)
        segment_ids.extend([0] * n_pad)

        # 5. pad the mask tokenst to the max length - output of transformer

        if max_mask > n_pred:
            n_pad = max_mask - n_pred
            masked_tokens.extend([0] * n_pad)
            masked_pos.extend([0] * n_pad)

        # 6. check whether is positive or negative

        if tokens_a_index + 1  == tokens_b_index and positive < batch_size /2 : # keep padding
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, True]) # input_ids, segment_ids -> inputs ,  masked_tokens, masked_pos -> outputs, True -> this is really the next sentence (label for the second loss)
            positive += 1
        elif tokens_a_index + 1 != tokens_b_index and negative < batch_size /2 : # keep padding
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, False]) # input_ids, segment_ids -> inputs ,  masked_tokens, masked_pos -> outputs, True -> this is really the next sentence (label for the second loss)
            negative += 1
        
    return batch

In [10]:
batch = make_batch()

In [11]:
len(batch)

6

In [12]:
input_ids, segment_ids, masked_tokens, masked_pos, isNetx = map(torch.LongTensor, zip(*batch))

In [13]:
input_ids.shape, segment_ids.shape, masked_tokens.shape, masked_pos.shape, isNetx

(torch.Size([6, 1000]),
 torch.Size([6, 1000]),
 torch.Size([6, 5]),
 torch.Size([6, 5]),
 tensor([0, 1, 1, 0, 1, 0]))

In [14]:
masked_tokens

tensor([[150,  25, 169, 157, 117],
        [181, 136, 122, 142,  76],
        [ 60,  53, 144, 189, 150],
        [111,  28, 115, 104, 170],
        [180, 164, 139,  34, 121],
        [129,  32, 149,  34, 164]])

# Model

In [15]:
class Embedding(nn.Module):
    def __init__(self):
        super(Embedding, self).__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model) # token embedding
        self.pos_embed = nn.Embedding(max_len, d_model) # position embedding
        self.seg_embed = nn.Embedding(n_segments, d_model) # segment (token types) embedding
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, seg):
        # x, seg:(bs, len)
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long, device=x.device)
        pos = pos.unsqueeze(0).expand_as(x) # (len,) -> (bs, len)
        embedding = self.tok_embed(x) + self.pos_embed(pos) + self.seg_embed(seg)
        return self.norm(embedding)


# Attention mask

In [16]:
def get_attn_pad_mask(seq_q, seq_k):
    batch_size, len_q = seq_q.size()
    batch_size, len_k = seq_k.size()

    # eq(zero) is PAD token
    pad_attn_mask = seq_k.data.eq(0).unsqueeze(1) # batch_size x 1 x len_k(=len_q), one is masking
    return pad_attn_mask.expand(batch_size, len_q, len_k) # batch_size x len_k x len_q

In [17]:
# Testing the attention
print(get_attn_pad_mask(input_ids, input_ids).shape)

torch.Size([6, 1000, 1000])


# Encoder

The encoder ahs two main components:
Multi-head attention
position-wise feed-forward network

very simialr to transformer

In [18]:
class EncoderLayer(nn.Module):
    def __init__(self):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention()
        self.pos_ffn = PoswiseFeedForwardNet()

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask)
        enc_outputs = self.pos_ffn(enc_outputs) # enc_outputs: [batch_size x len_q x d_model]
        return enc_outputs, attn

In [19]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(d_k)
        scores.masked_fill_(attn_mask, -1e9) # fill masked positions with large negative value
        attn = nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        return context, attn


In [20]:
# paramters definition
n_layer = 6 # number of encoder of encoder layer
n_heads = 8 # number of heads in multi head attention
d_model = 768 # embedding size
d_ff = d_model * 4  # 4 *d_model, FeedForward dimension
d_k = d_v = 64 # dimension of the k(=Q), V
n_segments = 2

In [21]:
class MultiHeadAttention(nn.Module):
    def __init__(self):
        super(MultiHeadAttention, self).__init__()
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)
        self.fc = nn.Linear(n_heads * d_v, d_model)

    def forward(self, Q, K, V, attn_mask):
        # q: [batch_size x len_q x d_model], k: [batch_size x len_k x d_model], v: [batch_size x len_v x d_model]
        residual, batch_size = Q, Q.size(0)

        # (B, S, D) -> (B, H, S, W)
        q_s = self.W_Q(Q).view(batch_size, -1, n_heads, d_k).transpose(1, 2)
        k_s = self.W_K(K).view(batch_size, -1, n_heads, d_k).transpose(1, 2)
        v_s = self.W_V(V).view(batch_size, -1, n_heads, d_v).transpose(1, 2)

        attn_mask = attn_mask.unsqueeze(1).repeat(1, n_heads, 1, 1)

        context, attn = ScaledDotProductAttention()(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, n_heads * d_v)
        output = self.fc(context)
        return nn.LayerNorm(d_model)(output + residual), attn


In [22]:
class PoswiseFeedForwardNet(nn.Module):

    def __init__(self):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # (batch_size, len_seq, d_model) -> (batch_size, len_seq, d_ff) -> (batch_size, len_seq, d_model)
        return self.fc2(F.gelu(self.fc1(x)))


# put them tpgether

In [23]:
class BERT(nn.Module):

    def __init__(self):
        super(BERT, self).__init__()
        self.embedding = Embedding()
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(n_layer)])
        self.fc = nn.Linear(d_model, d_model)
        self.active = nn.Tanh()
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 2)

        # decoder is shared with embedding layer
        embed_weight = self.embedding.tok_embed.weight
        n_vocab, n_dim = embed_weight.size()
        self.decoder = nn.Linear(n_dim, n_vocab, bias=False)
        self.decoder.weight = embed_weight
        self.decoder_bias = nn.Parameter(torch.zeros(n_vocab))

    def forward(self, input_ids, segment_ids, masked_pos):
        output = self.embedding(input_ids, segment_ids)
        enc_self_attn_mask = get_attn_pad_mask(input_ids, input_ids)
        for layer in self.layers:
            output, enc_self_attn = layer(output, enc_self_attn_mask)

        # 1) predict next sentence using [CLS]
        h_pooled = self.active(self.fc(output[:, 0])) # [batch_size, d_model]
        logits_nsp = self.classifier(h_pooled) # [batch_size, 2]

        # 2) predict masked tokens
        masked_pos = masked_pos[:, :, None].expand(-1, -1, output.size(-1)) # [batch_size, max_pred, d_model]
        h_masked = torch.gather(output, 1, masked_pos)
        h_masked = self.norm(F.gelu(self.linear(h_masked)))
        logits_lm = self.decoder(h_masked) + self.decoder_bias # [batch_size, max_pred, n_vocab]

        return logits_lm, logits_nsp


# Training

In [25]:
num_epoch = 500
model = BERT()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNetx = map(torch.LongTensor, zip(*batch))

for epoch in range(num_epoch):
    optimizer.zero_grad()
    logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)
    # logits_lm: (bs, max_mask, vocab_size) ==> (6, 5, 34)
    # logits_nsp : (bs, yes/no) => (6, 2)
    
    # 1.mlm loss
    # logits_lm.transpose: (bs, vocab/_size, max_mask) vs. masked_tokens: (bs, max_masked)
    loss_lm = criterion(logits_lm.transpose(1, 2), masked_tokens) # for masked LM
    loss_lm = (loss_lm.float()).mean()

    # 2.nsp loss
    # logits_nsp : (bs, 2) vs. isNext(bs, )
    loss_nsp = criterion(logits_nsp, isNetx)

    # 3.combine loss
    loss = loss_lm + loss_nsp
    if epoch%100 == 0:
        print('Epoch:', '%02d' % (epoch), 'loss = ', '{:.6f}'.format(loss))
    loss.backward()
    optimizer.step()




Epoch: 00 loss =  72.068489
Epoch: 100 loss =  6.123877
Epoch: 200 loss =  3.956085
Epoch: 300 loss =  3.955734
Epoch: 400 loss =  4.002150


# Inference

In [28]:
# predict the ,asked tokens and isNext
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map (torch.LongTensor, zip(batch[2]))
print([id2word[w.item()] for w in input_ids[0] if id2word[w.item()]] !='[PAD]')

logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)
# logits_lm: (bs, max_mask, vocab_size) ==> (1, 5, 34)
# logits_nsp : (bs, yes/no) => (1, 2)
    
#  predict the masked tokens
#  max the probability along the vocab dim (2), [1] is the indicies opf the max, and  [0] is the first value
logits_lm = logits_lm.data.max(2)[1][0].data.numpy()
# note that zero is padding that we add tothe masked_tokens
print('masked tokens (words) : ', [id2word[pos.item()] for pos in masked_tokens[0]])
print('masked tokens list : ', [pos.item() for pos in masked_tokens[0]])
print('masked tokens (words) : ', [id2word[pos.item()] for pos in logits_lm])
print('predict masked tokens list : ', [pos for pos in logits_lm])

# predict nsp
logits_nsp = logits_nsp.data.max(1)[1][0].data.numpy()
print(logits_nsp)
print('isNext : ', True if isNext else False)
print('predict isNext : ', True if logits_nsp else False)


True
masked tokens (words) :  ['this', 'to', 'more', 'powerful', 'problems']
masked tokens list :  [33, 17, 14, 135, 80]
masked tokens (words) :  ['a', 'a', 'a', 'a', 'a']
predict masked tokens list :  [np.int64(111), np.int64(111), np.int64(111), np.int64(111), np.int64(111)]
0
isNext :  False
predict isNext :  False
